In [ ]:
# DINOv3 features for "Lost in the Museum"
#
# Why this and not more resolution on DINOv2: DINOv3 improves over DINOv2 by
# +10.8 points on **Met** -- the Metropolitan Museum instance-retrieval benchmark,
# i.e. photographs of artworks matched to catalogue images. That is this task,
# not an analogy. (+7.6 on AmsterTime, ~+10.9 GAP on instance retrieval overall.)
#
# It also supersedes the ViT-g@770 idea. DINOv2's high-resolution adaptation
# stopped at 518, so 770 extrapolates its position embeddings; DINOv3 was adapted
# with global crops at {512, 768}, so 768 is INSIDE its training regime.
#
# Two DINOv3 specifics that will silently corrupt features if ignored:
#   * patch size is 16, not DINOv2's 14
#   * it uses REGISTER tokens, so patch tokens are NOT hidden[:, 1:]. We take the
#     LAST n_patches tokens, which is correct for any number of registers.
import glob, os, time
import numpy as np, torch
from pathlib import Path
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

Image.MAX_IMAGE_PIXELS = None
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", torch.cuda.get_device_name(0) if DEV == "cuda" else "CPU (too slow -- attach a GPU)")

# Run this notebook TWICE: SIZE=768 then SIZE=512. Both are inside DINOv3's
# adaptation range (global crops {512, 768}), so the second is a genuine second
# VIEW of the strongest model rather than naive multi-scale -- and the 3-block
# result (0.87583 -> 0.91610) showed that adding a strong block accelerates the
# width curve instead of saturating it. 512/16 = 32 -> 1024 tokens, ~2.3x cheaper.
SIZE  = 512
MODEL = "facebook/dinov3-vitl16-pretrain-lvd1689m"   # 0.3B; ViT-H+ if quota allows
# 4, not 8: at 768px the coarse grid is 48x48 = 2304 tokens and attention is
# quadratic in that. ViT-H+ (0.8B) with batch 8 will OOM a 16 GB T4; the loop
# below halves BATCH and retries rather than dying an hour into the run.
BATCH = 4
CKPT  = f"/kaggle/working/dinov3_{SIZE}_ckpt.npy"
DONE  = f"/kaggle/working/dinov3_{SIZE}_done.npy"

DATA = Path("/kaggle/input/competitions/lost-in-the-museum-f1/archive/kaggle_dataset/kaggle_dataset")
if not DATA.exists():
    hits = [d for d in glob.glob("/kaggle/input/**/", recursive=True)
            if glob.glob(os.path.join(d, "*.png"))]
    for h in hits[:10]:
        print("  ", h, len(glob.glob(os.path.join(h, "*.png"))), "png")
    assert hits, "no PNG directory under /kaggle/input"
    DATA = Path(max(hits, key=lambda h: len(glob.glob(os.path.join(h, "*.png")))))
paths = sorted(DATA.glob("*.png"))
print(len(paths), "images from", DATA)
assert len(paths) == 20000, f"expected 20000, got {len(paths)} -- indices would not line up"

In [ ]:
# Weights: HF download needs Internet ON and the DINOv3 licence accepted. If that
# fails, fall back to an attached Kaggle Model/Dataset -- discovering this 20
# minutes into a booked GPU window is how a window gets lost.
from transformers import AutoModel

# DINOv3 is a GATED repo: the weights are free and the licence permits commercial
# use, but you must accept it on the model page AND authenticate. Three routes,
# tried in order, because a 401 forty minutes into a GPU window is expensive.
#
#   1. Kaggle Secret named HF_TOKEN  (Add-ons -> Secrets)
#   2. environment variable HF_TOKEN
#   3. an attached Kaggle Model / Dataset -- no auth needed at all
#
# For 1 or 2 you must first click "Agree and access repository" at
# https://huggingface.co/facebook/dinov3-vitl16-pretrain-lvd1689m
TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("using HF token from Kaggle Secrets")
except Exception as e:
    TOKEN = os.environ.get("HF_TOKEN")
    print("Kaggle Secret unavailable:", repr(e)[:120])
    print("HF_TOKEN env var:", "set" if TOKEN else "not set")

model = None
if TOKEN:
    try:
        model = AutoModel.from_pretrained(MODEL, token=TOKEN)
        print("loaded from the hub:", MODEL)
    except Exception as e:
        print("hub load with token failed:", repr(e)[:300])

if model is None:
    local = [d for d in glob.glob("/kaggle/input/**/", recursive=True)
             if os.path.exists(os.path.join(d, "config.json"))]
    print("attached dirs containing config.json:")
    for d in local: print("   ", d)
    if not local:
        print("\nNOTHING USABLE. Pick one:")
        print("  a) accept the licence on the HF model page, create a HF token,")
        print("     add it as a Kaggle Secret named HF_TOKEN, re-run")
        print("  b) search Kaggle Models for 'dinov3' and attach it (no auth)")
        print("  c) fall back to an ungated backbone (see the note below)")
        raise SystemExit("no DINOv3 weights available")
    # Exclude ConvNeXt explicitly. Its path also contains "dinov3" and sorts
    # BEFORE "vith16plus", so with both mirrors attached this cell would load a
    # ConvNeXt and the ViT code below would read hidden[:, 0] off a 4-D feature
    # map -- wrong features, no error.
    # Both DINOv3 ViT mirrors match 'dinov3', and glob order is NOT sorted, so
    # picking [0] could silently load the ViT-L we already have and waste the
    # run. Prefer the largest variant explicitly: H+ > L > B > S.
    vits = sorted(d for d in local
                  if 'dinov3' in d.lower() and 'convnext' not in d.lower())
    for _tag in ('h16plus', 'vith', 'vitl', 'vitb', 'vits'):
        _m = [d for d in vits if _tag in d.lower()]
        if _m: vits = _m; break
    pick = vits or sorted(d for d in local if 'convnext' not in d.lower()) or sorted(local)
    assert pick, "attach a DINOv3 ViT mirror (not the ConvNeXt one)"
    model = AutoModel.from_pretrained(pick[0].rstrip("/"))
    print("loaded from", pick[0])

model = model.eval().to(DEV)
# False, not True: fp16 returned ALL-NaN features for DINOv3 ViT-L at 512 --
# 40,960,000 NaNs, which then propagated silently through the concat. The
# ViT-H+@768 run was fine in fp16, so flip this back for that one if rerun.
HALF = False
if DEV == "cuda" and HALF:
    model = model.half()
assert getattr(model.config, "model_type", "") != "convnext" and \
       hasattr(model.config, "patch_size"), \
       f"loaded a non-ViT model ({getattr(model.config,'model_type','?')}) -- use kaggle-dinov3-convnext.ipynb instead"
PATCH = getattr(model.config, "patch_size", 16)
HID   = getattr(model.config, "hidden_size", 1024)
NPATCH = (SIZE // PATCH) ** 2
print(f"patch={PATCH} hidden={HID} tokens={NPATCH} (+cls +registers)")

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
class Imgs(Dataset):
    def __init__(self, paths):
        self.paths = paths
        self.tf = transforms.Compose([
            transforms.Resize((SIZE, SIZE), interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        try:
            return self.tf(Image.open(self.paths[i]).convert("RGB")), i, True
        except Exception:
            return torch.zeros(3, SIZE, SIZE), i, False

In [ ]:
# CLS + GeM(p=3), matching the descriptor recipe that reached 0.86912. GeM sits
# between average pooling (p=1, detail washes out) and max pooling (p->inf, one
# patch dominates); p=3 lets a few distinctive regions lead without ignoring
# the rest.
P = 3.0

@torch.no_grad()
def embed(batch):
    h = model(pixel_values=batch).last_hidden_state          # (B, 1+R+NPATCH, HID)
    cls = h[:, 0]
    pat = h[:, -NPATCH:]                                     # robust to register count
    gem = pat.float().clamp(min=1e-6).pow(P).mean(1).pow(1.0 / P)
    return torch.cat([cls.float(), gem], 1)                  # (B, 2*HID)

feats = np.load(CKPT) if os.path.exists(CKPT) else np.zeros((len(paths), 2 * HID), np.float32)
done  = np.load(DONE) if os.path.exists(DONE) else np.zeros(len(paths), bool)
for pat_ in (f"dinov3_{SIZE}_ckpt.npy", f"dinov3_{SIZE}_done.npy"):        # resume from an attached partial run
    hits = sorted(glob.glob(f"/kaggle/input/**/{pat_}", recursive=True))
    if hits and not os.path.exists(CKPT):
        arr = np.load(hits[0])
        if pat_.endswith("ckpt.npy") and arr.shape == feats.shape: feats = arr
        if pat_.endswith("done.npy") and arr.shape == done.shape:  done = arr
print(f"resuming with {done.sum()}/{len(paths)} already embedded", flush=True)

todo = np.flatnonzero(~done)
dl = DataLoader(Imgs([paths[i] for i in todo]), batch_size=BATCH, num_workers=2, shuffle=False)
t0 = time.time(); n = 0
def embed_safe(x):
    """Halve the batch on OOM instead of losing the run."""
    try:
        return embed(x)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if len(x) == 1: raise
        h = len(x) // 2
        print(f"    OOM at batch {len(x)} -> splitting", flush=True)
        return torch.cat([embed_safe(x[:h]), embed_safe(x[h:])])

for x, idx, ok in dl:
    x = x.to(DEV)
    if DEV == "cuda" and HALF: x = x.half()
    v = embed_safe(x).cpu().numpy()
    if n == 0 and not np.isfinite(v).all():
        # fp16 overflows on some DINOv3 variants at some resolutions and returns
        # ALL-NaN features. That is invisible downstream until a submission scores
        # zero, so check the FIRST batch and stop here.
        raise SystemExit('First batch is non-finite -- fp16 overflow. '
                         'Set HALF=False in cell 1 (fp32, ~2x slower) and re-run.')
    for j, i in enumerate(idx.numpy()):
        feats[todo[i]] = v[j]; done[todo[i]] = bool(ok[j])
    n += len(idx)
    if n % (BATCH * 25) == 0:
        r = n / (time.time() - t0)
        print(f"  {done.sum()}/{len(paths)}  {r:.1f} img/s  ETA {(len(todo)-n)/r/60:.0f} min", flush=True)
        np.save(CKPT, feats); np.save(DONE, done)
np.save(CKPT, feats); np.save(DONE, done)
print(f"done in {(time.time()-t0)/60:.1f} min; embedded {done.sum()}/{len(paths)}")
np.save(f"/kaggle/working/features_dinov3_{SIZE}.npy", feats)
np.save("/kaggle/working/dinov3_names.npy", np.array([p.name for p in paths]))
print(f"wrote features_dinov3_{SIZE}.npy", feats.shape)

In [ ]:
# ---- fuse and write the submission ------------------------------------------
# The other half of the descriptor is the rotation-corrected ViT-L block. Its
# flip logic (which of the 3,439 candidates beat their upright score by margin
# 0.15) is deterministic, so it was computed once locally and is attached here as
# an array -- re-deriving it in this notebook would only add a transcription
# risk with no benefit.
LC = sorted(glob.glob("/kaggle/input/**/l518_rotcorrected.npy", recursive=True))
NM = sorted(glob.glob("/kaggle/input/**/feature_names.npy", recursive=True))
if not LC:
    print("attach l518_rotcorrected.npy as a Dataset. Currently attached:")
    for d in sorted(glob.glob("/kaggle/input/*")): print("   ", d)
    raise SystemExit("missing the ViT-L block")
Lc = np.load(LC[0]).astype(np.float64)
names = np.load(NM[0], allow_pickle=True) if NM else np.array([p.name for p in paths])
print("ViT-L block", Lc.shape, "| names", names.shape)
assert len(Lc) == len(feats) == 20000

In [ ]:
# NOTE: with a 3-block concat already at 0.91610, do the fusion LOCALLY with
# task1_concat.py --g/--extra instead of here -- this cell only builds the
# two-block version and would be a regression. Kept for reference.
FUSE_HERE = False
if FUSE_HERE:
    def l2(x, e=1e-12):
        return x / (np.linalg.norm(x, axis=1, keepdims=True) + e)
    
    def whiten(x, dim):
        """Whiten at FULL width. At full dimension PCA is only an orthogonal rotation,
        which cosine ignores, so the transform degenerates to pure whitening and
        discards nothing. Truncating this space to 1536 measured -0.041, and the
        leaderboard curve ACCELERATED at the point truncation stopped
        (2048 -> .8356, 3072 -> .8456, 4096 -> .8523, 5120 full -> .8691)."""
        mu = x.mean(0, keepdims=True)
        _, s, vt = np.linalg.svd(x - mu, full_matrices=False)
        sc = (s[:dim] / np.sqrt(len(x) - 1)) + 1e-8
        return l2((x - mu) @ vt[:dim].T / sc).astype(np.float32)
    
    # per-block L2 BEFORE concatenating: otherwise the block with the larger raw norm
    # dominates the dot product. Every earlier fusion attempt sat at 0.80201 because
    # 8192 raw VLAD dims swamped 1536 DINOv2 ones -- unnormalised blocks, not
    # ensembling, were the problem.
    D3 = np.load("/kaggle/working/features_dinov3.npy").astype(np.float64)
    X = np.hstack([l2(D3), l2(Lc)])
    print("concat", X.shape)
    Z = whiten(X, X.shape[1])          # FULL width, never truncated
    print("whitened", Z.shape)
    
    import pandas as pd
    PREC = 5     # at ~4096 dims a unit component is ~0.016, so 5 decimals keeps 3
                 # significant digits; accumulated cosine error ~4e-4, far below the
                 # ~0.01 gaps that decide a ranking. Keeps the file upload-sized.
    df = pd.DataFrame(np.round(Z, PREC), columns=[f"feature_{i}" for i in range(Z.shape[1])])
    df.insert(0, "image_name", names)
    df["ID"] = names
    assert len(df) == 20000 and df.isna().sum().sum() == 0
    df.to_csv("/kaggle/working/submission.csv", index=False)
    print(f"wrote submission.csv: {len(df)} rows x {df.shape[1]} cols")